# 05 · Train all three models

One notebook, three runs, identical data and hyperparameters everywhere except
what is being tested.

| Run | Starts from | Trains on | Purpose |
|---|---|---|---|
| **Stage 1** | `nllb600m-guz-init` (nb 04) | `stage1.jsonl` | learn Ekegusii — **the baseline** |
| **Stage 2** | **stage 1** | `stage2.jsonl` (PSA + 25% replay) | adapt to PSAs — **the result** |
| **Mixed** | `nllb600m-guz-init` | `mixed.jsonl` | control: was the ordering worth it? |

Stage 2 starts from the stage-1 weights, which is the whole point of a
curriculum. It also uses a **lower learning rate** — the model already knows
Ekegusii and we are nudging its register, not teaching it a language from
scratch. A high LR here is the fastest way to destroy what stage 1 built.

**Runtime** — roughly 3–5 h for stage 1, 1–2 h for stage 2, 4–6 h for the mixed
control. Each run checkpoints and can be resumed; you can also run stage 1 and
stage 2, look at notebook 06, and come back for the control.

**Inputs** — `artifacts/data/*.jsonl`, `artifacts/nllb600m-guz-init/`
**Outputs** — three model directories under `artifacts/`

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
sys.path.insert(0, str(pathlib.Path.cwd()))
import nb_common as C

C.set_seed()
C.use_house_style()
print(f"project root: {C.ROOT}")

In [ ]:
import json, math, torch, numpy as np
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq, EarlyStoppingCallback)

C.gpu_report()
C.require_files(C.DATA / "stage1.jsonl", C.DATA / "stage2.jsonl",
                C.DATA / "mixed.jsonl", C.DATA / "dev.jsonl")
assert C.EXTENDED_MODEL.exists(), "Run notebook 04 first"
MIX = json.loads((C.DATA / "mixture.json").read_text())
print("\nplanned runs:", {k: f"{v:,}" for k, v in MIX["counts"].items()})

## 1. Hyperparameters

Two independent learning rates. `LR_STAGE1` teaches a new language and can be
the usual 5e-5. `LR_STAGE2` is deliberately 3× smaller: the model already speaks
Ekegusii and stage 2 only has ~23k examples, so a large LR would overwrite
stage 1 rather than refine it.

Lower `BATCH` if you hit out-of-memory — the node is shared, so size against the
*free* VRAM printed above.

In [ ]:
# --- memory-aware configuration ------------------------------------------------
# This node is SHARED: ~66 GiB of the 79 GiB card belongs to other tenants, so
# plan against what is actually free, which changes minute to minute.
#
# Two things dominate memory here, and neither is the model:
#
#  1. OPTIMIZER STATE. Full AdamW needs ~4.6 GiB of moments on top of 4.6 GiB of
#     weights and gradients - a 9.2 GiB floor before any activation. 8-bit Adam
#     cuts that to 5.7 GiB, Adafactor to 4.7 GiB, at negligible cost for a
#     fine-tune this size.
#
#  2. THE LOGITS TENSOR. NLLB's vocabulary is 256k and accelerate upcasts model
#     outputs to fp32, so ONE example costs 128 x 256206 x 4 = 125 MB of logits,
#     250 MB once the gradient is counted.
#
# LABEL SMOOTHING IS OFF, and that is the important line in this cell. HF's
# LabelSmoother computes `-log_softmax(logits)` in Python, materialising a second
# full-size fp32 copy of the logits plus its gradient - doubling the per-example
# cost from 250 MB to 500 MB. Setting the factor to 0 makes the model use its own
# fused CrossEntropyLoss instead, which allocates no such duplicate. The cost is
# perhaps 0.3 BLEU; the benefit is that training fits on a contended card.

import os, math, torch

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

MAX_LEN         = 128
EFFECTIVE_BATCH = 48
LR_STAGE1       = 5e-5
LR_STAGE2       = 1.5e-5
EPOCHS_S1       = 3
EPOCHS_S2       = 3
WARMUP          = 0.03
LABEL_SMOOTH    = 0.0     # see above - 0 disables HF's memory-hungry smoother
EVAL_EVERY      = 500

RUN_STAGE1 = True
RUN_STAGE2 = True
RUN_MIXED  = False        # leave off until stage 1 and 2 have finished

def pick_optimizer():
    try:
        import bitsandbytes  # noqa: F401
        return "adamw_bnb_8bit", 5.7
    except ImportError:
        return "adafactor", 4.7

OPTIM, OPTIM_FLOOR_GIB = pick_optimizer()

VOCAB = 256_206
bytes_per_example = MAX_LEN * VOCAB * 4 * 2          # fp32 logits + gradient
if LABEL_SMOOTH:
    bytes_per_example *= 2                            # smoother duplicates both
per_example_gib = bytes_per_example / 2**30

free_gib = torch.cuda.mem_get_info()[0] / 2**30 if torch.cuda.is_available() else 0.0
headroom = max(0.0, free_gib - OPTIM_FLOOR_GIB - 1.0)  # 1 GiB margin

BATCH = max(1, min(8, int(headroom / per_example_gib)))
ACCUM = max(1, round(EFFECTIVE_BATCH / BATCH))

print(f"free VRAM        : {free_gib:5.1f} GiB   (shared card - varies minute to minute)")
print(f"optimizer        : {OPTIM} (~{OPTIM_FLOOR_GIB:.1f} GiB floor)")
print(f"label smoothing  : {LABEL_SMOOTH}")
print(f"per example      : {per_example_gib*1024:5.0f} MB of logits")
print(f"headroom         : {headroom:5.1f} GiB")
print(f"batch {BATCH} x accum {ACCUM} = effective {BATCH * ACCUM}")

if headroom < 1.0:
    print("\n  !! Almost no headroom. Options, in order of preference:")
    print("     - wait for the other jobs to finish (nvidia-smi to watch)")
    print("     - MAX_LEN = 96      -> 25% less logits memory")
    print("     - ask Kinesis for a less contended node")
if OPTIM == "adafactor":
    print("\n  tip: pip install bitsandbytes -> 8-bit Adam, closer to AdamW quality")

## 2. Dataset

Special tokens are built by hand — `[lang_code] … [eos]` on both sides — as in
notebook 04. The label sequence therefore begins with the target-language token,
which is exactly what `forced_bos_token_id` produces at inference.

Upsampling was already materialised in notebook 03, so this class is a plain
encoder with no sampling logic hidden inside it.

In [ ]:
tok = AutoTokenizer.from_pretrained(C.EXTENDED_MODEL)
EOS = tok.eos_token_id

def load_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(l) for l in fh]

class TranslationDataset(Dataset):
    def __init__(self, rows, max_len=MAX_LEN):
        self.rows, self.max_len = rows, max_len

    def _enc(self, text, lang):
        ids = tok(text, add_special_tokens=False, truncation=True,
                  max_length=self.max_len - 2)["input_ids"]
        return [tok.convert_tokens_to_ids(lang)] + ids + [EOS]

    def __len__(self): return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        return {"input_ids": self._enc(r["src"], r["src_lang"]),
                "labels":    self._enc(r["tgt"], r["tgt_lang"])}

dev_ds = TranslationDataset(load_jsonl(C.DATA / "dev.jsonl"))
ex = TranslationDataset(load_jsonl(C.DATA / "stage1.jsonl"))[0]
print(f"dev examples: {len(dev_ds):,}")
print("source:", tok.decode(ex["input_ids"], skip_special_tokens=False)[:90])
print("target:", tok.decode(ex["labels"], skip_special_tokens=False)[:90])

## 3. One training function, used three times

Keeping the runs in a single function is what makes the comparison meaningful:
the only things that differ between them are the starting weights, the data and
the learning rate. Everything else is held constant by construction.

In [ ]:
import inspect, dataclasses, torch

class Seq2SeqCollator:
    """
    Pads a batch and guarantees `decoder_input_ids` exist.

    Why this is needed: with `label_smoothing_factor` set, the Trainer POPS
    `labels` out of the batch and computes the loss itself. The model therefore
    never sees labels, and if the batch has no `decoder_input_ids` it has
    nothing to shift - producing the misleading error
    "You cannot specify both decoder_input_ids and decoder_inputs_embeds"
    (that check actually fires when BOTH are None).

    transformers v5's DataCollatorForSeq2Seq no longer builds them, so we do it
    here. If a future version starts providing them again, this leaves them
    alone.
    """

    def __init__(self, tokenizer, model):
        self.base = DataCollatorForSeq2Seq(tokenizer, model=model,
                                           label_pad_token_id=-100,
                                           pad_to_multiple_of=8)
        self.pad_id = model.config.pad_token_id
        self.start_id = model.config.decoder_start_token_id

    def __call__(self, features):
        batch = self.base(features)
        if "decoder_input_ids" not in batch and "labels" in batch:
            labels = batch["labels"]
            shifted = labels.new_zeros(labels.shape)
            shifted[:, 1:] = labels[:, :-1].clone()
            shifted[:, 0] = self.start_id
            shifted.masked_fill_(shifted == -100, self.pad_id)
            batch["decoder_input_ids"] = shifted
        return batch


def make_training_args(**kwargs):
    """Drop arguments this transformers version no longer accepts, and say so."""
    supported = set(inspect.signature(Seq2SeqTrainingArguments.__init__).parameters)
    try:
        supported |= {f.name for f in dataclasses.fields(Seq2SeqTrainingArguments)}
    except Exception:
        pass
    unknown = sorted(k for k in kwargs if k not in supported)
    if unknown:
        print(f"  note: transformers {__import__('transformers').__version__} "
              f"does not support {unknown} - dropped")
    return Seq2SeqTrainingArguments(**{k: v for k, v in kwargs.items() if k in supported})


def train_run(name, data_file, init_from, lr, epochs, out_dir):
    print("=" * 70)
    print(f"  {name}")
    print(f"  init: {pathlib.Path(init_from).name}  data: {data_file}  lr: {lr}")
    print("=" * 70)

    rows = load_jsonl(C.DATA / data_file)
    train_ds = TranslationDataset(rows)
    model = AutoModelForSeq2SeqLM.from_pretrained(init_from)
    model.config.max_length = MAX_LEN
    collator = Seq2SeqCollator(tok, model)

    # warmup_ratio is deprecated in v5; derive explicit warmup_steps instead.
    total_steps = math.ceil(len(train_ds) / (BATCH * ACCUM)) * epochs
    warmup_steps = max(1, int(WARMUP * total_steps))

    args = make_training_args(
        output_dir=str(C.ARTIFACTS / "checkpoints" / name),
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=ACCUM,
        learning_rate=lr,
        optim=OPTIM,
        auto_find_batch_size=True,   # halve the batch and retry on OOM
        num_train_epochs=epochs,
        warmup_steps=warmup_steps,
        label_smoothing_factor=LABEL_SMOOTH,
        weight_decay=0.01,
        lr_scheduler_type="linear",
        bf16=torch.cuda.is_available(),
        gradient_checkpointing=True,
        group_by_length=True,      # dropped automatically where unsupported
        logging_steps=100,
        eval_strategy="steps", eval_steps=EVAL_EVERY,
        save_strategy="steps", save_steps=EVAL_EVERY,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss", greater_is_better=False,
        predict_with_generate=False,
        report_to="none", seed=C.SEED,
    )

    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds,
                             eval_dataset=dev_ds, data_collator=collator,
                             callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

    # Prove the batch is well formed before committing hours to it.
    probe = collator([train_ds[i] for i in range(2)])
    assert "decoder_input_ids" in probe, "collator failed to build decoder_input_ids"
    assert (probe["decoder_input_ids"][:, 0] == model.config.decoder_start_token_id).all()
    print(f"  batch keys: {sorted(probe)}")
    print(f"  {len(train_ds):,} examples, ~{total_steps:,} steps, "
          f"{warmup_steps:,} warmup")

    result = trainer.train()
    trainer.save_model(str(out_dir))
    tok.save_pretrained(out_dir)
    print(f"  saved -> {out_dir}")
    return trainer, result

## 4. Stage 1 — learn Ekegusii

Starts from the tokenizer-extended checkpoint, where `guz_Latn` exists but has
only Kikuyu's embedding. This run is where Ekegusii is actually learned, and its
output is the **baseline** every later number is compared against.

In [ ]:
history = {}
if RUN_STAGE1:
    t1, r1 = train_run("stage1", "stage1.jsonl", str(C.EXTENDED_MODEL),
                       LR_STAGE1, EPOCHS_S1, C.STAGE1_MODEL)
    history["stage1"] = t1.state.log_history
    print(r1.metrics)
else:
    print("skipped (RUN_STAGE1 = False)")

## 5. Stage 2 — adapt to PSA register

Starts from **stage 1's weights**, not from the base model. Gentle learning
rate, 25% Bible replay in the data. Watch the dev loss: if it rises sharply the
LR is too high for this phase.

In [ ]:
if RUN_STAGE2:
    assert C.STAGE1_MODEL.exists(), "Stage 1 must run first - stage 2 continues from it"
    t2, r2 = train_run("stage2", "stage2.jsonl", str(C.STAGE1_MODEL),
                       LR_STAGE2, EPOCHS_S2, C.STAGE2_MODEL)
    history["stage2"] = t2.state.log_history
    print(r2.metrics)
else:
    print("skipped (RUN_STAGE2 = False)")

## 6. Mixed control — everything at once

Same data as stage 1 and stage 2 combined, same starting point as stage 1, but
in a single pass with no ordering. If the two-stage model beats this one, the
curriculum earned its place; if not, the simpler recipe is the honest answer.

In [ ]:
if RUN_MIXED:
    t3, r3 = train_run("mixed", "mixed.jsonl", str(C.EXTENDED_MODEL),
                       LR_STAGE1, EPOCHS_S1, C.MIXED_MODEL)
    history["mixed"] = t3.state.log_history
    print(r3.metrics)
else:
    print("skipped (RUN_MIXED = False) - notebook 06 will evaluate whatever exists")

## 7. Loss curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(history) or 1, figsize=(5 * max(1, len(history)), 4),
                         squeeze=False)
for ax, (name, hist) in zip(axes[0], history.items()):
    tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
    ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
    if tr: ax.plot(*zip(*tr), color=C.PALETTE[0], label="train")
    if ev: ax.plot(*zip(*ev), color=C.PALETTE[1], marker="o", label="dev")
    ax.set_title(name); ax.set_xlabel("step"); ax.set_ylabel("cross-entropy"); ax.legend()
plt.tight_layout(); C.save_fig(fig, "05_loss_curves"); plt.show()

C.save_json({name: [{k: v for k, v in h.items() if k in ("step", "loss", "eval_loss")}
                    for h in hist] for name, hist in history.items()},
            C.DATA / "training_history.json")
print("Next: 06_evaluate.ipynb")